# 10.1 Semantic meaning with embeddings

Explore pretrained word and document embeddings. Add your own code and markdown cells after each prompt and before its checks. Preserve supplied cells and IDs. All 100 points are awarded manually: 50 code and 50 interpretation. Assertions check required results, not the quality of semantic conclusions. Supplied plotting and clustering code is not scored. Read the README.


In [ ]:
import sys
assert sys.version_info[:2] == (3, 13), "This activity requires Python 3.13. Select its virtual-environment kernel."

from pathlib import Path
from tempfile import gettempdir
import os
import csv
import gc

download_root = Path(gettempdir()) / "compint" / "10.1.embeddings"
for name in ("gensim", "sentence-transformers", "huggingface"):
    (download_root / name).mkdir(parents=True, exist_ok=True)
os.environ["GENSIM_DATA_DIR"] = str(download_root / "gensim")
os.environ["HF_HOME"] = str(download_root / "huggingface")
os.environ["HF_HUB_CACHE"] = str(download_root / "huggingface" / "hub")
# Configure directories BEFORE importing downloaders. Restart if already imported.
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import gensim.downloader as api
from sentence_transformers import SentenceTransformer, util
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, MeanShift

activity_dir = Path.cwd() if (Path.cwd() / "data/documents.csv").exists() else Path.cwd().parent
with (activity_dir / "data/documents.csv").open(encoding="utf-8", newline="") as file:
    records = list(csv.DictReader(file))
document_ids = [r["id"] for r in records]
documents = [r["text"] for r in records]
print("Downloaded assets:", download_root)
for doc_id, text in zip(document_ids, documents):
    print(doc_id, text)


Temporary caches may be removed by your operating system. First execution downloads models; later executions reuse cached files. Google News is about 1.74 GB compressed and needs several GB of RAM; GloVe-100 is about 134 MB compressed. Load one word model at a time, finish all its operations, retain only selected vectors/results, then delete the model and call `gc.collect()` before loading the other. Never commit models or caches.


In [ ]:
model_names = {"glove": "glove-wiki-gigaword-100", "word2vec": "word2vec-google-news-300"}
word_labels = ["king", "queen", "man", "woman", "dog", "cat", "horse", "bird", "car", "bus", "train", "bicycle", "paris", "france", "japan", "tokyo", "banana", "apple", "student", "teacher"]
word_pairs = [("king", "queen"), ("king", "banana"), ("car", "bus"), ("student", "teacher")]
capitalized = {"paris": "Paris", "france": "France", "japan": "Japan", "tokyo": "Tokyo"}
model_tokens = {"glove": word_labels, "word2vec": [capitalized.get(w, w) for w in word_labels]}


## 1. Word embeddings — code: 30 points

Add code that processes both models sequentially. Use `api.load(model_names[key])` and the tokens in `model_tokens[key]`. These retrieve pretrained vectors; do not train a model.

Create dictionaries keyed by `glove` and `word2vec`:
- **Loading and inspection (10):** `word_vectors[key]`, an array with one row per supplied word, in order. Print model name, vocabulary size, vector dimension, and matrix shape; show the shape of a single word vector without printing its values. Hints: `vector_size`, `len(model)`, `model[token]`, `np.stack`.
- **Similarities and neighbors (10):** `word_scores[key]`, four cosine scores in `word_pairs` order; `word_neighbors[key]`, the five nearest neighbors of `student`. The supplied Seaborn heatmap displays the pair scores. Print the labeled nearest-neighbor results. Hints: `similarity`, `most_similar`.
- **Analogies (10):** `word_analogies[key]`, a dictionary with `royalty` and `capitals`, each containing five `(word, score)` results. Evaluate **king − man + woman** and **Paris − France + Japan**, using the correct case per model. Print the operations and candidates. Hint: `most_similar(positive=[...], negative=[...], topn=5)`.

Retain these small results, delete each full model, then collect garbage. No exact neighbor or rank is required.


In [ ]:
assert "word_vectors" in globals(), "Create word_vectors."
for key, dimension in (("glove", 100), ("word2vec", 300)):
    assert np.asarray(word_vectors[key]).shape == (len(word_labels), dimension)
    assert np.isfinite(word_vectors[key]).all()
assert "word_scores" in globals() and "word_neighbors" in globals()
for key in model_names:
    scores = np.asarray(word_scores[key])
    assert scores.shape == (len(word_pairs),) and np.isfinite(scores).all()
    assert (np.abs(scores) <= 1.00001).all()
    assert len(word_neighbors[key]) == 5
assert "word_analogies" in globals()
for key in model_names:
    for analogy in ("royalty", "capitals"):
        assert len(word_analogies[key][analogy]) == 5


In [ ]:
def show_similarity_heatmap(scores, row_labels, column_labels, title):
    # Illustration only: the supplied scores were computed by the student.
    scores = np.asarray(scores)
    fig, ax = plt.subplots(figsize=(max(6, len(column_labels) * 0.55), max(4, len(row_labels) * 0.45)))
    sns.heatmap(scores, xticklabels=column_labels, yticklabels=row_labels,
                annot=True, fmt=".2f", cmap="vlag", vmin=-1, vmax=1, center=0,
                annot_kws={"size": 8}, cbar_kws={"label": "Cosine similarity"}, ax=ax)
    ax.set(title=title)
    ax.tick_params(axis="x", labelrotation=45)
    ax.tick_params(axis="y", labelrotation=0)
    fig.tight_layout()
    plt.show()

show_similarity_heatmap(
    np.column_stack([word_scores[key] for key in model_names]),
    [f"{a} / {b}" for a, b in word_pairs],
    ["GloVe", "Word2Vec"], "Word-pair cosine similarities")


**Interpretation Q1 — 10 points.** Explain rows and learned dimensions: are columns vocabulary words or named concepts? Use the supplied heatmap to cite two word comparisons and candidates from both analogies. What supports semantic structure? What differs between models? Why might training data, vocabulary, and bias affect results? Explain why an approximate analogy is not a universal rule and why static word vectors cannot distinguish the two senses of `bank` by context.

Add a markdown answer below.


## 2. Word visualization — supplied implementation

PCA projects learned features into two coordinates that preserve some variance, not all semantic relationships. It does not assign names to dimensions. Each model gets a separate projection; their axes and raw vectors are not interchangeable.


In [ ]:
def plot_embeddings(embeddings, labels, title):
    pca = PCA(n_components=2, svd_solver="full")
    coordinates = pca.fit_transform(np.asarray(embeddings))
    fig, ax = plt.subplots(figsize=(10, 7))
    ax.scatter(coordinates[:, 0], coordinates[:, 1])
    for label, point in zip(labels, coordinates):
        ax.annotate(label, point, xytext=(4, 4), textcoords="offset points")
    ax.set(xlabel="Principal component 1", ylabel="Principal component 2",
           title=f"{title} — variance retained: {pca.explained_variance_ratio_.sum():.1%}")
    ax.set_aspect("equal", adjustable="datalim")
    plt.show()
    return coordinates

for key in model_names:
    plot_embeddings(word_vectors[key], word_labels, model_names[key])


**Interpretation Q2 — 5 points.** Identify a visible group and an exception in each plot. Relate these to the word scores. How much variance was retained? Why might a close pair in the original space appear separated in 2D? Add a markdown answer.


## 3. Embed full documents — code: 10 points

Create `sentence_model` using `SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cpu", cache_folder=str(download_root / "sentence-transformers"))`. Use its `encode` method to create `document_embeddings` for `documents`, retaining input order and returning a NumPy array. Print model name, matrix shape, and the shape of one document vector with its document ID and text. Do not print raw embedding values.

This pretrained model produces 384 features per text. It is intended for sentences and short paragraphs; inputs exceeding its default 256 wordpiece limit are truncated. The supplied texts are short. Do not apply the earlier count/TF-IDF preprocessing: preserve natural text. No model training or averaging of static word vectors is requested.


In [ ]:
assert "document_embeddings" in globals(), "Create document_embeddings."
assert np.asarray(document_embeddings).shape == (len(documents), 384)
assert np.isfinite(document_embeddings).all()
assert (np.linalg.norm(document_embeddings, axis=1) > 0).all()


**Interpretation Q3 — 10 points.** Explain what a row represents and interpret the matrix and individual-vector shapes. Why do documents of different lengths have the same number of features? Contrast this with static word embeddings and bag of words/TF-IDF. How can context help with the supplied bank/python examples? What happens to a document beyond the model input limit? Add a markdown answer.


## 4. Compare documents — code: 10 points

Create `document_similarities` using `util.cos_sim` on the original `document_embeddings` against themselves. The result is a tensor; use `.detach().cpu().numpy()` to obtain `similarity_array` for display and sorting. The supplied Seaborn heatmap displays the matrix with document labels and annotated scores; you do not need to implement visualization. Warmer colors indicate higher similarity on the shared −1 to 1 scale. The diagonal shows self-similarity, so inspect off-diagonal cells to compare different documents.

Print scores and full texts for pairs D00/D01, D00/D06, D06/D18, and D00/D05. Create `nearest_document_ids` as the three integer row indices most similar to row 0, excluding row 0 itself. Display their document IDs, scores, and texts. Hint: sort the unrounded row in descending order.

Compute cosine in the original 384-dimensional space, before PCA. Scores range from −1 to 1 and are not probabilities. Similarity can reflect shared subject matter without agreement or logical equivalence.


In [ ]:
assert "document_similarities" in globals() and "similarity_array" in globals()
assert similarity_array.shape == (len(documents), len(documents))
assert np.isfinite(similarity_array).all()
assert np.allclose(similarity_array, similarity_array.T, atol=1e-5)
assert np.allclose(np.diag(similarity_array), 1, atol=1e-5)
assert (np.abs(similarity_array) <= 1.00001).all()
assert "nearest_document_ids" in globals()
assert len(nearest_document_ids) == len(set(nearest_document_ids)) == 3
assert all(isinstance(i, (int, np.integer)) and 0 < i < len(documents) for i in nearest_document_ids)
expected_ids = np.argsort(-similarity_array[0], kind="stable")
assert list(nearest_document_ids) == [int(i) for i in expected_ids if i != 0][:3]


In [ ]:
show_similarity_heatmap(similarity_array, document_ids, document_ids, "Document cosine similarities")


**Interpretation Q4 — 10 points.** Use the supplied heatmap to identify a high-similarity pair of different documents and a low-similarity pair. Cite their IDs and scores, as well as scores for the four specified pairs. Do paraphrases remain similar despite different words? Does shared `bank` vocabulary imply shared meaning? Does a high score imply agreement in the negation pair? Interpret the three retrieved documents and one surprising result. Add a markdown answer.


## 5. Visualize documents — supplied implementation

IDs connect points to the complete texts printed above. Keep these exact coordinates for every clustering comparison.


In [ ]:
document_xy = plot_embeddings(document_embeddings, document_ids, "Document embeddings")


**Interpretation Q5 — 5 points.** Describe visible groups and an ambiguous point using document IDs and text. Compare one plotted pair with its original-space cosine score. Explain why PCA coordinates are an illustration rather than proof of semantic similarity. Add a markdown answer.


## 6. Experiment with clustering — supplied implementation

KMeans asks you to supply `k`; it partitions the 2D points into that many groups. MeanShift seeks dense regions and does not accept a target cluster count. With `bandwidth=None` it estimates a neighborhood scale, so **it is not parameter-free**. Both operate on the same 2D projection here, not the original embedding space. Cluster IDs/colors do not correspond across runs.

Use the complete helper below in a **new code cell**, choosing at least three distinct `k` values between 2 and 23. For example, call `compare_clusters(document_xy, k_values=(2, 4, 6))`; choose and record your own values. You may make additional calls. No clustering implementation is required or scored. Optional: supply a positive `bandwidth` to explore MeanShift's scale sensitivity.


In [ ]:
def compare_clusters(points, k_values=(2, 4, 6), bandwidth=None):
    k_values = tuple(k_values)
    if len(set(k_values)) < 3 or any(not isinstance(k, int) or not 2 <= k < len(points) for k in k_values):
        raise ValueError("Choose at least three distinct integer k values from 2 to n_documents - 1.")
    results = {f"KMeans k={k}": KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(points)
               for k in k_values}
    results["MeanShift"] = MeanShift(bandwidth=bandwidth).fit_predict(points)
    fig, axes = plt.subplots(1, len(results), figsize=(6 * len(results), 6), sharex=True, sharey=True)
    for ax, (name, labels) in zip(axes, results.items()):
        groups = np.unique(labels)
        ax.scatter(points[:, 0], points[:, 1], c=labels, cmap="tab20")
        for doc_id, point in zip(document_ids, points):
            ax.annotate(doc_id, point, xytext=(3, 3), textcoords="offset points")
        ax.set(title=f"{name}: {len(groups)} clusters", xlabel="PC1", ylabel="PC2")
        ax.set_aspect("equal", adjustable="box")
        print(f"\n{name}: {len(groups)} clusters")
        for group in groups:
            indices = np.flatnonzero(labels == group)
            print(f"  Cluster {group}: {len(indices)} documents")
            for i in indices:
                print(f"    {document_ids[i]}: {documents[i]}")
    plt.show()
    return results


**Interpretation Q6 — 10 points.** Record your `k` values and MeanShift's resulting cluster count. Compare which documents merge or split as `k` changes (3 points). Explain how MeanShift can produce a cluster count without a supplied `k`, including bandwidth's role (3 points). Propose names and cite specific document memberships to compare MeanShift with KMeans (2 points). Which grouping is more useful for this corpus, and why? Discuss projection effects and why no result establishes a true number of semantic categories (2 points). A single MeanShift cluster or mixed groups are valid findings when supported by evidence.

Add your comparison call and markdown answer below. No numeric quality threshold or expected cluster membership is required.
